# What If? Scenarios

Simulating changes in price, demand, costs and profit. Analysing the inventory and the waste, simulating a reduction of waste,
estimating how many people the demand needs, looking at the economic context (inflation and wages) and testing combined scenarios.


In [ ]:
from pathlib import Path
from google.colab import files

REQUIRED_FILES = ['monthly_profit.csv', 'inventory.csv', 'economic_indicators.csv', 'hourly_sales.csv']
missing = [name for name in REQUIRED_FILES if not Path('/content', name).is_file()]
if missing:
    print('Select these CSV files:', ', '.join(missing))
    uploaded = files.upload()
else:
    print('All required CSV files are already uploaded.')

still_missing = [name for name in REQUIRED_FILES if not Path('/content', name).is_file()]
if still_missing:
    raise FileNotFoundError('Missing CSV files: ' + ', '.join(still_missing))
print('Ready! All required CSV files are present.')

All required CSV files are already uploaded.
Ready! All required CSV files are present.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('/content') if Path('/content').is_dir() else Path.cwd()
FIGURES = PROJECT_ROOT / 'reports' / 'figures'
TABLES = PROJECT_ROOT / 'reports' / 'tables'
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

def read_data(folder, filename):
    path = PROJECT_ROOT / filename
    if not path.is_file():
        raise FileNotFoundError(f'Missing {filename}. Run the upload cell at the top of this notebook.')
    df = pd.read_csv(path)
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
    return df

def save_plot(filename):
    plt.tight_layout()
    plt.savefig(FIGURES / filename, dpi=150, bbox_inches='tight')
    plt.close()

def save_table(df, filename):
    df.to_csv(TABLES / filename, index=False)

def check_close(actual, expected, label, tolerance=0.06):
    if not np.isclose(actual, expected, atol=tolerance):
        raise ValueError(f'{label}: {actual:.2f} != {expected:.2f}')

def quality_report(datasets):
    for name, df in datasets.items():
        print(f'{name}: {len(df)} rows; missing={df.isna().sum().sum()}; duplicates={df.duplicated().sum()}')

pd.set_option('display.float_format', '{:,.2f}'.format)
print('Working folder:', PROJECT_ROOT)


def baseline_numbers(monthly):
    revenue = monthly['revenue'].sum()
    variable_cost = monthly['variable_cost'].sum()
    fixed_cost = monthly[['rent', 'salaries', 'utilities', 'other_costs']].sum().sum()
    fees = monthly['royalty'].sum() + monthly['marketing_fee'].sum()
    return {'revenue': revenue, 'variable_cost': variable_cost,
            'fixed_cost': fixed_cost, 'fee_rate': fees/revenue,
            'profit': monthly['operating_profit'].sum()}

def simulate(base, label, price=0, demand=0, variable_unit_cost=0, fixed_delta=0):
    if min(1+price, 1+demand, 1+variable_unit_cost) < 0:
        raise ValueError('Negative price, demand or unit cost is not supported')
    new_revenue = base['revenue'] * (1+price) * (1+demand)
    new_variable = base['variable_cost'] * (1+demand) * (1+variable_unit_cost)
    profit = new_revenue * (1-base['fee_rate']) - new_variable - base['fixed_cost'] - fixed_delta
    return {'scenario': label, 'revenue_ron': new_revenue,
            'operating_profit_ron': profit, 'profit_change_ron': profit-base['profit'],
            'margin_pct': profit/new_revenue*100 if new_revenue else np.nan}

def waste_summary(inventory, reduction=.20):
    df = inventory.groupby(['item','unit'],as_index=False).agg(
        consumed=('consumed','sum'), waste=('waste','sum'), purchased=('purchased','sum'))
    df['waste_rate_pct'] = 100*df['waste']/(df['consumed']+df['waste']).replace(0,np.nan)
    df['waste_after_reduction'] = df['waste']*(1-reduction)
    df['units_saved'] = df['waste']*reduction
    return df

def staff_needed(hourly, minutes_per_item=2.5, productive_fraction=.80):
    if not 0 < productive_fraction <= 1:
        raise ValueError('Productive fraction must be between 0 and 1')
    df = hourly.copy()
    df['staff_needed'] = np.ceil(df['quantity']*minutes_per_item/(60*productive_fraction)).astype(int)
    return df

def max_demand_loss(base, price_increase):
    per_unit_demand_contribution = (base['revenue'] * (1+price_increase) *
                                    (1-base['fee_rate']) - base['variable_cost'])
    if per_unit_demand_contribution <= 0:
        return np.nan
    required_multiplier = (base['profit']+base['fixed_cost']) / per_unit_demand_contribution
    return max(0, (1-required_multiplier)*100)


Working folder: /content


## 1. Baseline

In [ ]:
monthly = read_data("processed", "monthly_profit.csv")
inventory = read_data("simulated", "inventory.csv")
economic = read_data("raw", "economic_indicators.csv")
hourly = read_data("processed", "hourly_sales.csv")

base = baseline_numbers(monthly)
for key, value in base.items():
    print(f"{key}: {value:,.4f}" if key == "fee_rate" else f"{key}: {value:,.2f} RON")

# the scenario function with no change must give exactly the real profit
check = simulate(base, "Base")
assert np.isclose(check["operating_profit_ron"], base["profit"], atol=0.06)
print("\nScenario baseline matches the financial statement.")

revenue: 150,943.00 RON
variable_cost: 52,441.93 RON
fixed_cost: 60,800.00 RON
fee_rate: 0.0550
profit: 29,399.20 RON

Scenario baseline matches the financial statement.


## 2. Business scenarios
In the function, **demand** changes both revenue and raw material cost, **price** changes only revenue,
and the royalty + marketing fee follow revenue.

In [ ]:
scenarios = [
    simulate(base, "Base"),
    simulate(base, "Ingredients +10%", variable_unit_cost=0.10),
    simulate(base, "Price +5%, same demand", price=0.05),
    simulate(base, "Price +5%, demand -5%", price=0.05, demand=-0.05),
    simulate(base, "Demand +10%", demand=0.10),
    simulate(base, "Fixed costs +1,500 RON", fixed_delta=1500),
    simulate(base, "Ingredients +10%, demand -5%", variable_unit_cost=0.10, demand=-0.05),
]
comparison = pd.DataFrame(scenarios)
comparison

,scenario,revenue_ron,operating_profit_ron,profit_change_ron,margin_pct
0,Base,"150,943.00","29,399.20",0.00,19.48
1,Ingredients +10%,"150,943.00","24,155.01","-5,244.19",16.00
2,"Price +5%, same demand","158,490.15","36,531.26","7,132.06",23.05
3,"Price +5%, demand -5%","150,565.64","31,664.69","2,265.49",21.03
4,Demand +10%,"166,037.30","38,419.12","9,019.92",23.14
5,"Fixed costs +1,500 RON","150,943.00","27,899.20","-1,500.00",18.48
6,"Ingredients +10%, demand -5%","143,395.85","19,907.26","-9,491.94",13.88


In [ ]:
comparison.plot.barh(x="scenario", y="profit_change_ron", figsize=(11, 5), legend=False,
                     title="Operating Profit Change by Scenario (3 months)", color="saddlebrown")
plt.xlabel("Change in profit (RON)")
plt.ylabel("")
plt.axvline(0, color="black", linewidth=0.8)
save_plot("scenarios_comparison.png")

## 3. Pricing: how much demand can we lose?
Without knowing the customers' reaction, I test a grid of price increases and demand changes.

In [ ]:
price_changes = [0, 0.025, 0.05, 0.075, 0.10]
demand_changes = [-0.20, -0.15, -0.10, -0.05, 0, 0.05]

grid = []
for p in price_changes:
    for d in demand_changes:
        row = simulate(base, f"Price {p:+.1%}, Demand {d:+.0%}", price=p, demand=d)
        row["price_change_pct"] = p * 100
        row["demand_change_pct"] = d * 100
        grid.append(row)
pricing = pd.DataFrame(grid)

# table: rows = price change, columns = demand change, values = profit change in RON
pricing.pivot(index="price_change_pct", columns="demand_change_pct", values="profit_change_ron").round(0)

demand_change_pct,-20.00,-15.00,-10.00,-5.00,0.00,5.00
price_change_pct,,,,,,
0.00,"-18,040.00","-13,530.00","-9,020.00","-4,510.00",0.00,"4,510.00"
2.50,"-15,187.00","-10,499.00","-5,810.00","-1,122.00","3,566.00","8,254.00"
5.00,"-12,334.00","-7,468.00","-2,601.00","2,265.00","7,132.00","11,999.00"
7.50,"-9,481.00","-4,437.00",608.00,"5,653.00","10,698.00","15,743.00"
10.00,"-6,629.00","-1,405.00","3,818.00","9,041.00","14,264.00","19,487.00"


In [ ]:
for p in price_changes:
    part = pricing[pricing["price_change_pct"] == p * 100]
    plt.plot(part["demand_change_pct"], part["profit_change_ron"], marker="o", label=f"price {p:+.1%}")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Profit change for different price and demand changes")
plt.xlabel("Demand change (%)")
plt.ylabel("Profit change (RON, 3 months)")
plt.legend()
save_plot("scenarios_pricing_grid.png")

In [ ]:
# the break-even demand loss: the biggest fall in demand that still leaves profit at least as high as today
loss_table = pd.DataFrame({
    "price_increase_pct": [2.5, 5, 7.5, 10],
    "max_demand_loss_pct": [max_demand_loss(base, p) for p in [0.025, 0.05, 0.075, 0.10]],
})
loss_table

,price_increase_pct,max_demand_loss_pct
0,2.50,3.80
1,5.00,7.33
2,7.50,10.60
3,10.00,13.65


## 4. Cost shocks

In [ ]:
macro_shocks = pd.DataFrame([
    simulate(base, f"Variable costs +{inc:.0%}", variable_unit_cost=inc) for inc in [0, 0.05, 0.10, 0.15]
])
macro_shocks

,scenario,revenue_ron,operating_profit_ron,profit_change_ron,margin_pct
0,Variable costs +0%,150943,"29,399.20",0.00,19.48
1,Variable costs +5%,150943,"26,777.10","-2,622.10",17.74
2,Variable costs +10%,150943,"24,155.01","-5,244.19",16.00
3,Variable costs +15%,150943,"21,532.91","-7,866.29",14.27


In [ ]:
# for context, the macro data for the months in the simulation
economic["month"] = economic["month"].astype(str)
economic_context = economic[economic["month"].isin(monthly["month"].astype(str))].copy()
economic_context

,month,annual_inflation_pct,avg_gross_wage_ron,avg_net_wage_ron
24,2026-01,9.60,9220,5518
25,2026-02,9.30,9272,5557
26,2026-03,9.90,9902,5938


*Inflation is not the same as the real change in the price of coffee ingredients, so the cost shocks above are only hypothetical.*

## 5. Inventory and waste

In [ ]:
inventory["date"] = pd.to_datetime(inventory["date"])

# the stock balance must work: opening + purchased - consumed - waste = closing
inventory["calculated_closing"] = (inventory["opening_stock"] + inventory["purchased"]
                                   - inventory["consumed"] - inventory["waste"])
inventory["balance_difference"] = inventory["closing_stock"] - inventory["calculated_closing"]
print("Maximum balance difference:", inventory["balance_difference"].abs().max())

Maximum balance difference: 2.2737367544323206e-13


In [ ]:
# waste in physical units, and what a 20% reduction would save
stock = waste_summary(inventory, reduction=0.20)
stock.round(3)

,item,unit,consumed,waste,purchased,waste_rate_pct,waste_after_reduction,units_saved
0,coffee_beans,kg,197.25,0.99,205.00,0.50,0.79,0.20
1,cups,pcs,"21,917.00",221.31,"22,100.00",1.00,177.05,44.26
2,milk,l,"2,515.82",75.90,"2,620.00",2.93,60.72,15.18


There are no prices per kg, per liter or per cup in the data, so I **cannot** turn these quantities into RON honestly.
The table shows the physical quantities only.

## 6. Staffing: how many people does the demand need?

In [ ]:
# my assumptions: 2.5 minutes of staff time per item, and 80% of each hour is productive time.
staffed = staff_needed(hourly, minutes_per_item=2.5, productive_fraction=0.80)

coverage = staffed.groupby("hour").agg(
    mean_items=("quantity", "mean"),
    peak_items=("quantity", "max"),
    peak_staff_needed=("staff_needed", "max"),
    share_hours_needing_2plus=("staff_needed", lambda s: (s >= 2).mean() * 100),
)
coverage

,mean_items,peak_items,peak_staff_needed,share_hours_needing_2plus
hour,,,,
7,19.16,35,2,46.67
8,34.42,56,3,98.89
9,26.97,41,3,86.67
10,20.02,32,2,50.00
11,16.64,30,2,28.89
12,19.66,39,3,45.56
13,19.86,35,2,54.44
14,14.74,27,2,14.44
15,14.48,25,2,10.00


In [ ]:
coverage["mean_items"].plot.bar(figsize=(10, 5), title="Average Hourly Demand", color="peru", rot=0)
plt.xlabel("Hour")
plt.ylabel("Average Items Sold")
save_plot("scenarios_staff_demand.png")

print("Share of all open hours that need...")
print((staffed["staff_needed"].value_counts(normalize=True).sort_index() * 100).round(1).rename("percent of hours"))

Share of all open hours that need...
staff_needed
1   59.50
2   37.90
3    2.60
Name: percent of hours, dtype: float64


This is a capacity illustration, not a work schedule. It does not include breaks, opening and closing tasks, or the fact that
someone has to be in the café all the opening hours.

## 7. Export the tables

In [ ]:
save_table(comparison, "scenarios_comparison.csv")
save_table(pricing, "scenarios_pricing_grid.csv")
save_table(macro_shocks, "scenarios_cost_shocks.csv")
save_table(stock, "scenarios_inventory_waste.csv")
save_table(coverage.reset_index(), "scenarios_staff_capacity.csv")
save_table(economic_context, "scenarios_economic_context.csv")

## What I found
Costs are the main risk: +10% on ingredients takes about 5,200 RON of profit in 3 months. Price is the best lever: a +5% price
can lose about 7% of the demand and the café still earns the same. Waste is small (milk is about 3%), so it will not change the profit much.
For staff, the morning is the pressure point: 08:00 and 09:00 need two or more people on most days, while about 60% of all open hours
could be handled by one person, so the schedule matters more than the number of people.